# Exercises and Business Applications

Add in more tools - perhaps to simulate actually booking a flight. A student has done this and provided their example in the community contributions folder.

Next: take this and apply it to your business. Make a multi-modal AI assistant with tools that could carry out an activity for your work. A customer support assistant? New employee onboarding assistant? So many possibilities! Also, see the week2 end of week Exercise in the separate Notebook.

## This is a combo of day 5 and the week2 EXERCISE. I went all out with this one

<p>This is a trading platform that helps monitor the price of stocks and a few major cryptocurrencies. It also supports paper trading
<p>

In [ ]:
# Business that sells stocks (doesn't actually sell stocks, just gives you prices lol)

import os
import json
import time
import requests
from datetime import date, timedelta, datetime
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import tempfile
import matplotlib
matplotlib.use("Agg")  # render to files, no pop-up windows
import matplotlib.pyplot as plt

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
massive_api_key = os.getenv("MASSIVE_API_KEY")
print("OpenAI key:", "set" if openai_api_key else "NOT set")
print("Massive key:", "set" if massive_api_key else "NOT set")

MODEL = "gpt-4.1-mini"
openai = OpenAI()

In [ ]:
BASE_URL = "https://api.massive.com"
HEADERS = {"Authorization": f"Bearer {massive_api_key}"}

In [ ]:
system_message = """You are a helpful stock market assistant. Keep answers short and clear.
Always use your tools for prices and history; never guess numbers.
Prices come from end-of-day data, so say 'last close' rather than 'current price'.
Confirm ticker and share count before any paper trade.
You provide information only, not financial advice."""

In [ ]:
_cache = {} # I use a free plan so this protects the 5 calls/minute limit

def _get(path, params=None, ttl=300):
    key = (path, json.dumps(params, sort_keys=True))
    if key in _cache and time.time() - _cache[key][0] < ttl:
        return _cache[key][1]
    r = requests.get(BASE_URL + path, headers=HEADERS, params=params, timeout=15)
    if r.status_code == 429:
        raise RuntimeError("Rate limit hit (free plan allows 5 calls/min). Try again shortly.")
    if r.status_code == 401:
        raise RuntimeError("Invalid or missing Massive API key.")
    r.raise_for_status()
    data = r.json()
    _cache[key] = (time.time(), data)
    return data

In [ ]:
latest_chart = None  # path to the most recent chart, picked up by the UI

def make_chart(ticker, dates, closes):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(dates, closes, linewidth=2)
    ax.fill_between(dates, closes, min(closes), alpha=0.15)
    ax.set_title(f"{ticker.upper()} daily close")
    ax.set_ylabel("USD")
    ax.grid(alpha=0.3)
    fig.autofmt_xdate()
    path = tempfile.NamedTemporaryFile(suffix=".png", delete=False).name
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    return path

In [ ]:
def get_price_history(ticker, days=30):
    global latest_chart
    end = date.today()
    start = end - timedelta(days=int(days))
    data = _get(f"/v2/aggs/ticker/{ticker.upper()}/range/1/day/{start}/{end}",
                {"adjusted": "true", "sort": "asc", "limit": 500})
    bars = data.get("results", [])
    if not bars:
        return f"No history found for {ticker}"
    closes = [b["c"] for b in bars]
    dates = [datetime.fromtimestamp(b["t"] / 1000) for b in bars]  # "t" is a ms timestamp
    latest_chart = make_chart(ticker, dates, closes)
    change = (closes[-1] - closes[0]) / closes[0] * 100
    return (f"{ticker.upper()} over {len(closes)} trading days: "
            f"start ${closes[0]:.2f}, end ${closes[-1]:.2f} ({change:+.1f}%), "
            f"high ${max(closes):.2f}, low ${min(closes):.2f}. A chart is shown to the user.")

In [ ]:
portfolio = {}  # {"AAPL": {"shares": 10, "cost": 1890.0}}, paper trading portfolio only tho

def buy_stock(ticker, shares):
    data = _get(f"/v2/aggs/ticker/{ticker.upper()}/prev", {"adjusted": "true"})
    if not data.get("results"):
        return f"Can't buy {ticker}: no price data"
    price = data["results"][0]["c"]
    pos = portfolio.setdefault(ticker.upper(), {"shares": 0, "cost": 0.0})
    pos["shares"] += shares
    pos["cost"] += shares * price
    return f"Paper trade: bought {shares} {ticker.upper()} at ${price} (total ${shares * price:,.2f})"


In [ ]:
def get_portfolio():
    if not portfolio:
        return "Portfolio is empty"
    return "; ".join(f"{t}: {p['shares']} shares, cost ${p['cost']:,.2f}" for t, p in portfolio.items())

In [ ]:
# This will help me build tool definitions, certainly very convenient haha

def make_tool(name, description, props, required):
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": props,
                       "required": required, "additionalProperties": False}}}

In [ ]:
# Uses my make_tool helper function from the above cell

tools = [
    make_tool("get_stock_price",
              "Get the most recent closing price and OHLC for a stock ticker symbol.",
              {"ticker": {"type": "string", "description": "Stock ticker, e.g. AAPL"}}, ["ticker"]),
    make_tool("get_price_history",
              "Get a summary of daily prices for a ticker over the last N days.",
              {"ticker": {"type": "string"},
               "days": {"type": "integer", "description": "Number of calendar days, e.g. 30"}},
              ["ticker"]),
    make_tool("buy_stock",
              "Simulate buying shares in a paper portfolio. Only call after the user explicitly confirms the ticker and share count.",
              {"ticker": {"type": "string"}, "shares": {"type": "integer"}}, ["ticker", "shares"]),
    make_tool("get_portfolio", "Show the user's current paper-trading holdings.", {}, []),
]

In [ ]:
TOOL_FUNCTIONS = {
    "get_stock_price": get_stock_price,
    "get_price_history": get_price_history,
    "buy_stock": buy_stock,
    "get_portfolio": get_portfolio,
}

In [ ]:
def handle_tool_calls(message):
    results = []
    for call in message.tool_calls:
        try:
            args = json.loads(call.function.arguments or "{}")
            result = TOOL_FUNCTIONS[call.function.name](**args)
        except Exception as e:
            result = f"Tool error: {e}"
        results.append({"role": "tool", "content": result, "tool_call_id": call.id})
    return results

In [ ]:
def transcribe(audio_path):
    if not audio_path:
        return ""
    with open(audio_path, "rb") as f:
        result = openai.audio.transcriptions.create(model="whisper-1", file=f)
    return result.text

In [ ]:
def run_chat(history):
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        messages.append(message)
        messages.extend(handle_tool_calls(message))
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    return response.choices[0].message.content

In [ ]:
def talker(text):
    response = openai.audio.speech.create(model="gpt-4o-mini-tts", voice="shimmer", input=text)
    path = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False).name
    with open(path, "wb") as f:
        f.write(response.content)
    return path

In [ ]:
def chat(history):
    global latest_chart
    latest_chart = None
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    reply = run_chat(history)
    history.append({"role": "assistant", "content": reply})
    chart = latest_chart if latest_chart else gr.update()  # keep the old chart if there's no new one
    return history, talker(reply), chart

In [ ]:
def add_message(message, history):
    if not message.strip():
        return "", history
    return "", history + [{"role": "user", "content": message}]
    

In [ ]:
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500)
        image_output = gr.Image(height=500, label="Chart")
    audio_output = gr.Audio(label="Assistant voice", autoplay=True)
    with gr.Row():
        entry = gr.Textbox(label="Ask about a stock (e.g. 'Show me Apple over 3 months'):", scale=3)
        mic = gr.Audio(sources=["microphone"], type="filepath", label="Or speak", scale=1)

    outputs = [chatbot, audio_output, image_output]

    # typed message
    entry.submit(add_message, [entry, chatbot], [entry, chatbot]).then(chat, chatbot, outputs)

    # spoken message: transcribe, then follow the same path as typing
    mic.stop_recording(transcribe, mic, entry).then(
        add_message, [entry, chatbot], [entry, chatbot]
    ).then(chat, chatbot, outputs)

ui.launch(inbrowser=True)

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a HUGE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>